# Superstore Sales & Profitability Analysis

**Name:** Anirudh Pawar
**Enrollment No.:** M24BBAU0266
**Program:** BBA, Bennett University

## Objective
Analyze four years of retail order data to understand which products, categories, and
regions drive revenue and profit, and identify the root cause behind any profitability
gaps using SQL.

**Dataset:** [Sample Superstore](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final) (Kaggle)
- 9,994 rows, 21 columns, covering Jan 2014 – Dec 2017
- Order/shipping details, customer segment, product category, sales, quantity, discount, profit


## 1. Load Data and Build a SQL Database

In [1]:
import pandas as pd
import sqlite3

# Load CSV (note the encoding - this file isn't standard UTF-8)
df = pd.read_csv("Sample - Superstore.csv", encoding="latin-1")

# Clean column names for SQL (remove spaces/hyphens)
df.columns = [c.strip().replace(" ", "_").replace("-", "_") for c in df.columns]

# Convert dates properly
df["Order_Date"] = pd.to_datetime(df["Order_Date"], format="%m/%d/%Y")
df["Ship_Date"] = pd.to_datetime(df["Ship_Date"], format="%m/%d/%Y")

# Create SQLite database and load the table
conn = sqlite3.connect("superstore.db")
df.to_sql("orders", conn, if_exists="replace", index=False)

print("Loaded", len(df), "rows into superstore.db")
conn.close()

Loaded 9994 rows into superstore.db


**Note:** the source CSV uses Latin-1 encoding rather than standard UTF-8, so it's
loaded with `encoding="latin-1"` to avoid decode errors. Column names are cleaned
(spaces/hyphens replaced with underscores) since SQL handles unquoted identifiers more
cleanly without them.

## 2. Core KPIs

In [2]:
conn = sqlite3.connect("superstore.db")

query = """
SELECT 
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    COUNT(DISTINCT Order_ID) AS Total_Orders,
    ROUND(AVG(Discount), 3) AS Avg_Discount
FROM orders
"""

result = pd.read_sql_query(query, conn)
print(result)

conn.close()

   Total_Sales  Total_Profit  Total_Orders  Avg_Discount
0   2297200.86     286397.02          5009         0.156


**Result:** $2,297,200.86 in total sales, $286,397.02 in total profit, across 5,009
unique orders, with an average discount of 15.6% per line item.

## 3. Top Products, Category, and Region Breakdown

In [3]:
conn = sqlite3.connect("superstore.db")

# Top 10 products by sales
top_products = """
SELECT Product_Name,
       ROUND(SUM(Sales), 2) AS Total_Sales,
       ROUND(SUM(Profit), 2) AS Total_Profit
FROM orders
GROUP BY Product_Name
ORDER BY Total_Sales DESC
LIMIT 10
"""
print("TOP 10 PRODUCTS BY SALES")
print(pd.read_sql_query(top_products, conn))

# Sales & profit by category
category_breakdown = """
SELECT Category,
       ROUND(SUM(Sales), 2) AS Total_Sales,
       ROUND(SUM(Profit), 2) AS Total_Profit,
       ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin_Pct
FROM orders
GROUP BY Category
ORDER BY Total_Sales DESC
"""
print("\nCATEGORY BREAKDOWN")
print(pd.read_sql_query(category_breakdown, conn))

# Sales & profit by region
region_breakdown = """
SELECT Region,
       ROUND(SUM(Sales), 2) AS Total_Sales,
       ROUND(SUM(Profit), 2) AS Total_Profit
FROM orders
GROUP BY Region
ORDER BY Total_Sales DESC
"""
print("\nREGION BREAKDOWN")
print(pd.read_sql_query(region_breakdown, conn))

conn.close()

TOP 10 PRODUCTS BY SALES
                                        Product_Name  Total_Sales  \
0              Canon imageCLASS 2200 Advanced Copier     61599.82   
1  Fellowes PB500 Electric Punch Plastic Comb Bin...     27453.38   
2  Cisco TelePresence System EX90 Videoconferenci...     22638.48   
3       HON 5400 Series Task Chairs for Big and Tall     21870.58   
4         GBC DocuBind TL300 Electric Binding System     19823.48   
5   GBC Ibimaster 500 Manual ProClick Binding System     19024.50   
6               Hewlett Packard LaserJet 3310 Copier     18839.69   
7  HP Designjet T520 Inkjet Large Format Printer ...     18374.90   
8          GBC DocuBind P400 Electric Binding System     17965.07   
9        High Speed Automatic Electric Letter Opener     17030.31   

   Total_Profit  
0      25199.93  
1       7753.04  
2      -1811.08  
3          0.00  
4       2233.51  
5        760.98  
6       6983.88  
7       4094.98  
8      -1878.17  
9       -262.00  

CATEGORY BREAKDO

**Key observations:**
- Several top-selling products are actually unprofitable or break-even: the Cisco
  TelePresence System (-$1,811 profit on $22.6K sales), HON 5400 Task Chairs ($0 profit
  on $21.9K sales), and the GBC DocuBind P400 (-$1,878 profit on $18K sales).
- **Furniture generates almost as much revenue as Technology** ($741,999.80 vs
  $836,154.03) but its profit margin (2.49%) is roughly **7x lower** than Technology
  (17.40%) and Office Supplies (17.04%).
- **Regionally**, West leads in both sales ($725,457.82) and profit ($108,418.45),
  followed by East. South has lower sales than Central but a *higher* profit —
  indicating South is more profit-efficient per dollar sold.

## 4. Root Cause: Discount Impact on Profit Margin

In [4]:
conn = sqlite3.connect("superstore.db")

discount_impact = """
SELECT Category,
       ROUND(AVG(Discount), 3) AS Avg_Discount,
       ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Profit_Margin_Pct
FROM orders
GROUP BY Category
ORDER BY Avg_Discount DESC
"""
print(pd.read_sql_query(discount_impact, conn))

conn.close()

          Category  Avg_Discount  Profit_Margin_Pct
0        Furniture         0.174               2.49
1  Office Supplies         0.157              17.04
2       Technology         0.132              17.40


**Result:** the pattern is consistent and clear — **Furniture carries the highest
average discount (17.4%)** of any category, and has by far the **lowest profit margin
(2.49%)**. Technology carries the lowest average discount (13.2%) and has the highest
margin (17.40%). Higher discounting directly tracks with lower profitability across all
three categories.

## Conclusion

- **Total Sales:** $2,297,200.86 | **Total Profit:** $286,397.02 | **5,009 orders** |
  15.6% average discount
- **Furniture is the key problem category**: comparable sales volume to Technology, but
  ~7x lower profit margin, driven by the highest average discount rate (17.4%) of any
  category.
- Several individual best-selling products (Cisco TelePresence, HON Task Chairs, GBC
  DocuBind P400) are breaking even or losing money despite strong sales — likely due to
  the same discounting pattern.
- **West region** is the strongest performer on both sales and profit; **South** is
  smaller in volume but more profit-efficient than Central.

## Business Recommendation
Reduce discounting on Furniture, particularly on high-volume items currently selling at
break-even or a loss. Technology and Office Supplies demonstrate that similar sales
volume is achievable at far better margins with lower discount rates — Furniture's
pricing/discount strategy should be brought closer in line with these categories.

## Dashboard
An interactive Power BI dashboard was built on this dataset, with KPI cards, a
Category vs. Profit Margin combo chart, a Region performance chart, and Category/Region
slicers for filtering. See `Superstore_Dashboard.pdf` / `dashboard_screenshot.png` in
this repository.
